# Mathematical & Architectural Guide to geLSTM and Its Ablations
### Longitudinal MCI $\to$ Alzheimer's Disease Conversion Prediction

---

## Executive Summary & Scope

The **Graph-Enhanced Long Short-Term Memory (geLSTM)** model is a hybrid spatio-temporal deep learning architecture tailored for early prediction of Alzheimer's Disease (AD) conversion from Mild Cognitive Impairment (MCI).

It processes **longitudinal resting-state fMRI functional connectivity (FC)** networks over multiple irregular clinical visits per subject, integrating:
1. A **Spatial Graph Attention Autoencoder (GAAE)** with Feature-wise Linear Modulation (**FiLM**) conditioning on demographics (age, sex) to compress $N_{\text{ROI}} \times N_{\text{ROI}}$ brain connectomes into low-dimensional latent embeddings $\mathbf{z}_t \in \mathbb{R}^d$.
2. A **Recurrent Core (LSTM / GRU)** that models sequential disease progression dynamics over variable-length visit sequences with inter-visit time intervals $\Delta t_t$.
3. A **Classification Head** that maps the final hidden recurrent state $\mathbf{h}_T$ to a conversion probability $P(\text{converter} \mid \text{trajectory})$.

To rigorously quantify the exact source of empirical gains and validate each inductive bias, the codebase implements **systematic ablations** across multiple orthogonal axes:
- **Graph Encoder & Pretraining Value** (`pretrained_frozen`, `pretrained_finetuned`, `random`, `none`)
- **Recurrent Core Inductive Bias** (LSTM vs. GRU / GEGRU)
- **Temporal Dynamics Modeling** (Active $\Delta t$ vs. Dropped $\Delta t$ vs. Zeroed $\Delta t$)
- **Sequence Permutation & Directionality** (Chronological order vs. `shuffle_order`)
- **Latent Space Filtering** (Full 64-d GAAE space vs. Top-$K$ FDR-selected dimensions)
- **Classifier Head Architecture & Normalization** (Standard Head vs. LayerNorm Head vs. Direct Linear)
- **Capacity & Network Depth Scaling** (1-layer vs. 2-layer, hidden size 16 to 128)
- **Sequence Length & Early Detection** (Full trajectory vs. First-$N$ visits with window controls)
- **Cross-Paradigm Comparison** (Recurrent geLSTM vs. Flattened GEC-MLP vs. Static Pooled GEP)

---

## 1. Problem Formulation & Base Mathematical Pipeline

### 1.1 Input Representation per Subject
Let subject $s \in \{1, \dots, S\}$ have $T_s$ longitudinal clinical visits ($T_s \in \{1, 2, \dots, T_{\max}\}$, typically $T_{\max} \le 6$). The longitudinal trajectory is represented as:

$$\mathcal{S}_s = \Big( (\mathcal{G}_{s, 1}, \Delta t_{s, 1}), (\mathcal{G}_{s, 2}, \Delta t_{s, 2}), \dots, (\mathcal{G}_{s, T_s}, \Delta t_{s, T_s}) \Big), \quad y_s \in \{0, 1\}$$

where:
- $y_s = 1$ denotes a **converter** (MCI converting to AD during the study window).
- $y_s = 0$ denotes **stable MCI**.
- Demographic covariates: $\mathbf{c}_s = [\text{sex}_s, \text{age}_{s, \text{norm}}]^\top \in \mathbb{R}^2$, where $\text{sex}_s \in \{0, 1\}$ and $\text{age}_{s, \text{norm}} \in [0, 1]$.
- Normalized inter-visit interval $\Delta t_{s, t} \in [0, 1]$ (normalised by $T_{\text{scale}} = 108.0$ months):

$$\Delta t_{s, t} = \begin{cases} 0.0, & t = 1 \\ \frac{\text{months}_{s, t} - \text{months}_{s, t-1}}{108.0}, & t > 1 \end{cases}$$

### 1.2 Per-Visit Brain Functional Connectivity Graph
At visit $t$, resting-state fMRI yields a pairwise Pearson correlation matrix $\mathbf{C}_t \in [-1, 1]^{N \times N}$ ($N=200$ ROIs from the Schaefer-200 atlas). The graph $\mathcal{G}_t = (\mathcal{V}, \mathbf{X}_t, \mathcal{E}_t, \mathbf{E}_t)$ is constructed via:
1. **Node Feature Matrix**: $\mathbf{X}_t = \mathbf{C}_t \in \mathbb{R}^{N \times D_{\text{in}}}$ where $D_{\text{in}} = N = 200$. Node $i$'s feature $\mathbf{x}_{t, i} \in \mathbb{R}^{200}$ is its whole-brain connectivity profile.
2. **Adjacency / Edges**: $\mathcal{E}_t$ is constructed as a directed $k$-nearest neighbor ($k=8$) graph based on functional similarity, with self-loops removed.
3. **Edge Attributes**: $\mathbf{e}_{t, ij} = [C_{t, ij}] \in \mathbb{R}^{1}$.

### 1.3 Graph Encoder: 3-Layer GATv2 with FiLM Modulation
The GAAE encoder $f_\phi(\mathcal{G}_t, \mathbf{c}_s)$ processes each visit graph independently.

#### Layer 1 & 2: GATv2 with Edge Features & Residual Connections
For layer $l \in \{1, 2\}$, with input node representation $\mathbf{h}_i^{(l-1)} \in \mathbb{R}^{D_{l-1}}$ and $K$ attention heads:

$$\alpha_{ij}^{(l, k)} = \frac{\exp\left(\text{LeakyReLU}\left(\mathbf{a}_l^{(k)\top} \left[ \mathbf{W}_l^{(k)} \mathbf{h}_i^{(l-1)} \,\|\, \mathbf{W}_l^{(k)} \mathbf{h}_j^{(l-1)} \,\|\, \mathbf{W}_{e, l}^{(k)} \mathbf{e}_{ij} \right]\right)\right)}{\sum_{j' \in \mathcal{N}(i) \cup \{i\}} \exp\left(\text{LeakyReLU}\left(\mathbf{a}_l^{(k)\top} \left[ \mathbf{W}_l^{(k)} \mathbf{h}_i^{(l-1)} \,\|\, \mathbf{W}_l^{(k)} \mathbf{h}_{j'}^{(l-1)} \,\|\, \mathbf{W}_{e, l}^{(k)} \mathbf{e}_{ij'} \right]\right)\right)}$$

$$\mathbf{h}_i^{(l)} = \text{BatchNorm}\left( \bigoplus_{k=1}^K \left( \sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij}^{(l, k)} \mathbf{W}_l^{(k)} \mathbf{h}_j^{(l-1)} + \mathbf{W}_{e, l}^{(k)} \mathbf{e}_{ij} \right) + \mathbf{W}_{\text{res}}^{(l)} \mathbf{h}_i^{(l-1)} \right)$$

#### Layer 3: Projection to Latent Bottleneck
Layer 3 uses head averaging without concatenation:
$$\mathbf{z}_i^{(3)} = \frac{1}{K} \sum_{k=1}^K \left( \sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij}^{(3, k)} \mathbf{W}_3^{(k)} \mathbf{h}_j^{(2)} + \mathbf{W}_{e, 3}^{(k)} \mathbf{e}_{ij} \right) + \mathbf{W}_{\text{res}}^{(3)} \mathbf{h}_i^{(2)} \in \mathbb{R}^{d} \quad (d = 64)$$

#### Feature-wise Linear Modulation (FiLM)
Demographic conditioning $\mathbf{c}_s \in \mathbb{R}^2$ modulates the latent representations via affine scale $\boldsymbol{\gamma}$ and shift $\boldsymbol{\beta}$:

$$\boldsymbol{\gamma}(\mathbf{c}_s) = \mathbf{W}_{\gamma, 2} \text{ReLU}(\mathbf{W}_{\gamma, 1} \mathbf{c}_s + \mathbf{b}_{\gamma, 1}) + \mathbf{b}_{\gamma, 2} \in \mathbb{R}^d$$

$$\boldsymbol{\beta}(\mathbf{c}_s) = \mathbf{W}_{\beta, 2} \text{ReLU}(\mathbf{W}_{\beta, 1} \mathbf{c}_s + \mathbf{b}_{\beta, 1}) + \mathbf{b}_{\beta, 2} \in \mathbb{R}^d$$

$$\tilde{\mathbf{z}}_{t, i} = \boldsymbol{\gamma}(\mathbf{c}_s) \odot \mathbf{z}_i^{(3)} + \boldsymbol{\beta}(\mathbf{c}_s)$$

#### Graph-Level Readout & Per-Fold Standardisation
Mean-pooling across all $N=200$ brain regions:

$$\mathbf{z}_t = \frac{1}{N} \sum_{i=1}^N \tilde{\mathbf{z}}_{t, i} \in \mathbb{R}^d$$

Z-score standardisation using training-fold statistics $\boldsymbol{\mu}_{\text{train}}, \boldsymbol{\sigma}_{\text{train}} \in \mathbb{R}^d$:

$$\hat{\mathbf{z}}_t = \frac{\mathbf{z}_t - \boldsymbol{\mu}_{\text{train}}}{\boldsymbol{\sigma}_{\text{train}} + \epsilon} \in \mathbb{R}^d$$

### 1.4 Sequential Recurrent Processing & Classification
1. **Recurrent Input Construction**:
   $$\mathbf{u}_t = [\hat{\mathbf{z}}_t \,\|\, \Delta t_t] \in \mathbb{R}^{d+1} \quad (64 + 1 = 65)$$
2. **Recurrent Hidden State Transition**:
   $$(\mathbf{h}_t, \mathbf{c}_t) = \text{LSTM}(\mathbf{u}_t, \mathbf{h}_{t-1}, \mathbf{c}_{t-1}) \quad \text{for } t=1, \dots, T_s$$
3. **Terminal State Readout**: $\mathbf{h}_{T_s} \in \mathbb{R}^H$ (last hidden state corresponding to the final observed visit).
4. **Classifier Head & Logit**:
   $$z_{\text{logit}} = \mathbf{w}_2^\top \text{Dropout}\Big(\text{ReLU}\big(\mathbf{W}_1 \mathbf{h}_{T_s} + \mathbf{b}_1\big)\Big) + b_2 \in \mathbb{R}$$
   $$\hat{P}(\text{converter} \mid \mathcal{S}_s) = \sigma(z_{\text{logit}}) = \frac{1}{1 + e^{-z_{\text{logit}}}}$$

---

## 2. Ablation Axis 1: Graph Encoder & Reconstruction-Value (`encoder_init`)

The fundamental question addressed by this ablation is: **How much value does self-supervised GAAE reconstruction pretraining provide to downstream AD conversion classification?**

Three distinct mechanisms could be driving downstream performance:
1. The **Reconstruction Pretraining** (representations learned from unlabelled graph autoencoding).
2. The **Encoder Architecture** (GATv2 + FiLM as an inductive bias, even with random weights).
3. The **Encoder Module Existence** (vs. directly feeding raw pooled ROI profiles to the LSTM).

### Diagram: Input Pipeline & Encoder Pathways

![Figure 1: Input Pipeline & Encoder Ablations](fig1_gelstm_encoder_and_input_ablations.png)

---

### 2.1 Mathematical & Architectural Comparison of the 4 Arms

| Attribute | Arm 1: `pretrained_frozen` (Reference) | Arm 2: `pretrained_finetuned` | Arm 3: `random` | Arm 4: `none` |
| :--- | :--- | :--- | :--- | :--- |
| **Encoder Module** | 3-layer GATv2 + FiLM | 3-layer GATv2 + FiLM | 3-layer GATv2 + FiLM | **None** (`model.encoder is None`) |
| **Weight Loading** | Pretrained GAAE checkpoint | Pretrained GAAE checkpoint | **No checkpoint** (Random Init) | **No checkpoint** |
| **Encoder Training** | **Frozen** ($\nabla_{\phi} \mathcal{L} = 0$) | **Trainable** ($\nabla_{\phi} \mathcal{L} \neq 0$) | **Trainable** ($\nabla_{\phi} \mathcal{L} \neq 0$) | N/A (No encoder) |
| **Per-Visit Embedding $\mathbf{z}_t$** | $\mathbf{z}_t = \text{Pool}(\text{GAAE}(\mathcal{G}_t)) \in \mathbb{R}^{64}$ | $\mathbf{z}_t = \text{Pool}(\text{GAAE}(\mathcal{G}_t)) \in \mathbb{R}^{64}$ | $\mathbf{z}_t = \text{Pool}(\text{GAAE}(\mathcal{G}_t)) \in \mathbb{R}^{64}$ | $\mathbf{z}_t = \text{Pool}(\mathbf{X}_t) \in \mathbb{R}^{200}$ |
| **Embedding Width $d_{\text{embed}}$** | $64$ (`gaae_latent`) | $64$ (`gaae_latent`) | $64$ (`gaae_latent`) | $200$ (`in_features`) |
| **LSTM Input Dim $D_{\text{in}}$** | $64 + 1 = 65$ | $64 + 1 = 65$ | $64 + 1 = 65$ | $200 + 1 = 201$ |
| **Active Downstream Params ($H=32, H_{\text{head}}=32$)** | **13,761** (LSTM + Head) | **303,985** (Encoder + LSTM + Head) | **303,985** (Encoder + LSTM + Head) | **31,169** (LSTM + Head) |
| **Scientific Hypothesis Isolated** | Baseline performance | Value of task-specific adaptation | Value of reconstruction pretraining | Value of graph encoder as a whole |

### 2.2 Deep Dive into Mathematical Differences Across Arms

#### Arm 4: `encoder_init="none"` (Direct Raw Feature Pooling)
Under `none`, the model constructs **zero encoder parameters**. Instead of passing through GAT convolutions and FiLM conditioning:

$$\mathbf{z}_t = \frac{1}{N} \sum_{i=1}^N \mathbf{x}_{t, i} = \frac{1}{200} \sum_{i=1}^{200} \mathbf{C}_{t, i, :} \in \mathbb{R}^{200}$$

Per-fold standardisation is applied directly over the 200 raw correlation dimensions:

$$\hat{\mathbf{z}}_t = \frac{\mathbf{z}_t - \boldsymbol{\mu}_{\text{raw}}}{\boldsymbol{\sigma}_{\text{raw}} + \epsilon} \in \mathbb{R}^{200}$$

The recurrent core input is automatically derived:

$$D_{\text{in}} = 200 + 1 = 201 \implies \mathbf{u}_t = [\hat{\mathbf{z}}_t \,\|\, \Delta t_t] \in \mathbb{R}^{201}$$

This isolates whether graph neural convolutions add any predictive advantage over direct longitudinal modeling of mean regional connectivity.

#### Arm 3: `encoder_init="random"` (Random Architecture Inductive Bias)
Under `random`, the identical GATv2 + FiLM module tree is built, but checkpoint loading is skipped (`load_gaae_weights` is guarded). Parameters are randomly initialised from standard distributions (Xavier/Kaiming):

$$\phi \sim \mathcal{P}_{\text{init}}$$

The entire model (encoder + recurrent core + head) is trained end-to-end with gradients flowing through graph attention operations into node transformations.

#### Arm 2: `pretrained_finetuned` vs. Arm 1: `pretrained_frozen`
- `pretrained_frozen`: $\phi$ is initialised from GAAE $\phi^*$ and held fixed: $\frac{\partial \mathcal{L}}{\partial \phi} \equiv 0$. In evaluation and training, visits are embedded inside `torch.no_grad()`.
- `pretrained_finetuned`: $\phi$ is initialised from GAAE $\phi^*$ but gradients are enabled:

$$\phi^{(k+1)} = \phi^{(k)} - \eta \nabla_\phi \mathcal{L}_{\text{BCE}}(y_s, \hat{y}_s)$$

This tests if task adaptation improves upon frozen self-supervised features or causes catastrophic overfitting given small cohort sizes ($N \approx 133$ subjects).

---

### 2.3 Gradient Flow & Memory Mechanics (`encoder_grad`)

In standard evaluation, `encode_batch_sequences` processes each visit under `torch.no_grad()` and an `eval_mode` context manager:
```python
# CLASSIFIER/model/GELSTM/utils.py
mode_ctx = nullcontext() if encoder_grad else eval_mode(encoder_model)
grad_ctx = nullcontext() if encoder_grad else torch.no_grad()

with mode_ctx, grad_ctx:
    # encode visits...
```
When `encoder_init` is `pretrained_finetuned` or `random`, `EvalConfig.encoder_grad = True` is activated during training, allowing PyTorch's dynamic computational graph to retain intermediate activations for GAT attention weights and FiLM layers across all $T_s$ visits per subject.

---

## 3. Ablation Axis 2: Recurrent Core Architecture (GELSTM vs. GEGRU)

The longitudinal conversion task operates on relatively short clinical trajectories ($T_s \le 6$ visits over 5 years). A central question in sequence modeling on small medical cohorts ($N \approx 133$ training subjects) is **model capacity vs. sample size**.

### 3.1 Mathematical Formulations

Given recurrent input $\mathbf{u}_t = [\hat{\mathbf{z}}_t \,\|\, \Delta t_t] \in \mathbb{R}^{D_{\text{in}}}$ and hidden state $\mathbf{h}_{t-1} \in \mathbb{R}^H$:

#### geLSTM (4-Gate Cell, Dual State)

$$\begin{aligned}
\mathbf{f}_t &= \sigma\left(\mathbf{W}_f \mathbf{u}_t + \mathbf{U}_f \mathbf{h}_{t-1} + \mathbf{b}_f\right) && \text{(Forget Gate: what to drop from memory cell)} \\
\mathbf{i}_t &= \sigma\left(\mathbf{W}_i \mathbf{u}_t + \mathbf{U}_i \mathbf{h}_{t-1} + \mathbf{b}_i\right) && \text{(Input Gate: what new information to store)} \\
\tilde{\mathbf{c}}_t &= \tanh\left(\mathbf{W}_c \mathbf{u}_t + \mathbf{U}_c \mathbf{h}_{t-1} + \mathbf{b}_c\right) && \text{(Candidate Cell State)} \\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t && \text{(Updated Memory Cell State)} \\
\mathbf{o}_t &= \sigma\left(\mathbf{W}_o \mathbf{u}_t + \mathbf{U}_o \mathbf{h}_{t-1} + \mathbf{b}_o\right) && \text{(Output Gate: what to expose to hidden state)} \\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t) && \text{(Updated Hidden State)}
\end{aligned}$$

#### geGRU (3-Gate Cell, Single State)

$$\begin{aligned}
\mathbf{r}_t &= \sigma\left(\mathbf{W}_r \mathbf{u}_t + \mathbf{U}_r \mathbf{h}_{t-1} + \mathbf{b}_r\right) && \text{(Reset Gate: how much past state to forget)} \\
\mathbf{z}_t^{\text{gate}} &= \sigma\left(\mathbf{W}_z \mathbf{u}_t + \mathbf{U}_z \mathbf{h}_{t-1} + \mathbf{b}_z\right) && \text{(Update Gate: balances past state vs. candidate)} \\
\tilde{\mathbf{h}}_t &= \tanh\left(\mathbf{W}_h \mathbf{u}_t + \mathbf{U}_h (\mathbf{r}_t \odot \mathbf{h}_{t-1}) + \mathbf{b}_h\right) && \text{(Candidate Hidden State)} \\
\mathbf{h}_t &= (1 - \mathbf{z}_t^{\text{gate}}) \odot \mathbf{h}_{t-1} + \mathbf{z}_t^{\text{gate}} \odot \tilde{\mathbf{h}}_t && \text{(Linear Interpolation State Update)}
\end{aligned}$$

---

### 3.2 Parameter Complexity & Capacity Comparison

Let $D_{\text{in}} = 65$ (64 GAAE latent + 1 $\Delta t$), $H = 32$, and $L = 1$ layer:

$$\text{Params}_{\text{LSTM}} = 4 \times \big(D_{\text{in}} \cdot H + H^2 + H\big) = 4 \times (65 \times 32 + 32^2 + 32) = 4 \times (2080 + 1024 + 32) = 12,672$$

$$\text{Params}_{\text{GRU}} = 3 \times \big(D_{\text{in}} \cdot H + H^2 + H\big) = 3 \times (65 \times 32 + 32^2 + 32) = 3 \times 3136 = 9,408$$

**Key Takeaways:**
1. **25% Parameter Reduction**: The GRU eliminates the separate cell state $\mathbf{c}_t$ and reduces 4 affine transforms to 3.
2. **Inductive Fit for Short Trajectories**: On sequences of length $T \le 6$, the additive cell-state gradient highway in LSTM provides little advantage over GRU's convex combination state update, while GRU's reduced parameter count lowers variance and risk of constant-predictor collapse.

---

## 4. Ablation Axis 3: Inter-Visit Time Delta Dynamics ($\Delta t$)

Clinical follow-up in Alzheimer's studies (such as DELCODE) has irregular intervals between scans (e.g. baseline, 12 months, 24 months, or skipped visits at 36 months). Modeling visit spacing is essential to separate acute vs. chronic trajectory changes.

### 4.1 Mathematical Comparison of Temporal Modes

#### Mode 1: Active Time Delta (`use_time_delta=True`, `zero_time_delta=False`)

$$\Delta t_t = \begin{cases} 0.0, & t = 1 \\ \frac{\text{months}_t - \text{months}_{t-1}}{108.0} \in [0, 1], & t > 1 \end{cases}$$

$$\mathbf{u}_t = \left[ \hat{\mathbf{z}}_t \,\|\, \Delta t_t \right] \in \mathbb{R}^{d+1} \quad (d+1 = 65)$$

The recurrent input affine projection learns a distinct temporal weighting column $\mathbf{w}_{\Delta t} \in \mathbb{R}^{4H}$:

$$\mathbf{W} \mathbf{u}_t = \mathbf{W}_z \hat{\mathbf{z}}_t + \mathbf{w}_{\Delta t} \Delta t_t$$

#### Mode 2: Dropped Time Delta (`use_time_delta=False`, `zero_time_delta=False`)

$$\mathbf{u}_t = \hat{\mathbf{z}}_t \in \mathbb{R}^d \quad (d = 64)$$

The model treats all inter-visit transitions as uniform ordinal steps ($t=1, 2, 3\dots$), discarding actual elapsed calendar time.

#### Mode 3: Zeroed Time Delta (`zero_time_delta=True`)

$$\mathbf{u}_t = \left[ \hat{\mathbf{z}}_t \,\|\, 0.0 \right] \in \mathbb{R}^{d+1}$$

Maintains the identical tensor dimensions ($d+1=65$) as Mode 1, but zeros out the time interval signal. This enables evaluating an already-trained model with $\Delta t$ ablated at test time without dimension mismatch.

---

## 5. Ablation Axis 4: Sequence Permutation & Temporal Directionality (`shuffle_order`)

A critical sanity check in longitudinal medical ML is verifying whether the recurrent model is genuinely exploiting **directional trajectory dynamics** or simply functioning as a permutation-invariant **bag-of-visits** / scan-count detector.

### 5.1 Permutation Formulation
For a subject with $T_s$ visits, let $\pi$ be a random permutation of $\{1, 2, \dots, T_s\}$:

$$\mathcal{S}_s^{\text{shuffled}} = \Big( (\mathcal{G}_{\pi(1)}, \Delta t_{\pi(1)}), (\mathcal{G}_{\pi(2)}, \Delta t_{\pi(2)}), \dots, (\mathcal{G}_{\pi(T_s)}, \Delta t_{\pi(T_s)}) \Big)$$

Notice that **$\Delta t$ is permuted alongside its corresponding visit graph**, preserving the marginal distribution of $(\mathcal{G}, \Delta t)$ pairs while completely destroying the monotonic arrow of time ($t_1 < t_2 < \dots < t_T$).

### 5.2 Hypothesis Testing Matrix
- If $\text{AUC}_{\text{ordered}} \gg \text{AUC}_{\text{shuffled}}$: The model relies critically on progressive temporal ordering (e.g. progressive hippocampal atrophy or deteriorating default mode network connectivity).
- If $\text{AUC}_{\text{ordered}} \approx \text{AUC}_{\text{shuffled}}$: The model is insensitive to order, utilizing the visits merely as multiple noisy samples or counting visit quantity.

---

## 6. Ablation Axis 5: Latent Dimensionality & FDR Feature Selection

The full GAAE latent space produces $d = 64$ continuous dimensions per visit. On small cohorts ($N \approx 100$ in-fold training subjects), feeding all 64 dimensions into recurrent weights can lead to overfitting on non-discriminative autoencoder reconstruction artifacts.

### 6.1 Mathematical Formulation of FDR Selection
For each latent feature $k \in \{1, \dots, 64\}$, compute the two-sample Fisher / Welch statistic between converters and stable MCI across the **training fold subjects only**:

$$t_k = \frac{\bar{z}_{k, \text{conv}} - \bar{z}_{k, \text{mci}}}{\sqrt{\frac{s_{k, \text{conv}}^2}{n_{\text{conv}}} + \frac{s_{k, \text{mci}}^2}{n_{\text{mci}}}}}, \quad p_k = 2 \left(1 - \Phi(|t_k|)\right)$$

Apply False Discovery Rate (Benjamini-Hochberg) or rank-order top-$K$ selection ($K \in \{8, 16, 32\}$):

$$\mathcal{I}_{\text{top}} = \arg\text{top-}K_{k \in \{1..64\}} (|t_k|)$$

### 6.2 Recurrent Core Re-patching
The visit embedding is sliced prior to recurrent ingestion:

$$\mathbf{z}_{t, \text{filtered}} = \mathbf{z}_t[\mathcal{I}_{\text{top}}] \in \mathbb{R}^K$$

$$\mathbf{u}_t = [\mathbf{z}_{t, \text{filtered}} \,\|\, \Delta t_t] \in \mathbb{R}^{K+1} \quad (16 + 1 = 17)$$

In `CLASSIFIER/adapters/gelstm.py`, `_patch_recurrent_core` dynamically reconstructs the LSTM/GRU with `input_size = K + 1`:

$$\text{Params}_{\text{LSTM}}(\text{FDR-}16) = 4 \times (17 \times 32 + 32^2 + 32) = 6,400 \quad (\approx 50\% \text{ reduction vs. Full 64-d})$$

---

## 7. Ablation Axis 6: Classifier Head Architecture & Normalization

Longitudinal RNN models trained on small cohorts often suffer from **probability squashing** (narrow output probability distributions concentrated around the base rate). The classifier head design directly controls logit sharpness and gradient backpropagation.

### 7.1 Mathematical Formulations of Head Variants

#### Variant A: Standard Head (`classifier_norm="none"`, `classifier_hidden=32`)

$$z_{\text{logit}} = \mathbf{w}_2^\top \text{Dropout}\left(\text{ReLU}\left(\mathbf{W}_1 \mathbf{h}_{T_s} + \mathbf{b}_1\right)\right) + b_2$$

where $\mathbf{W}_1 \in \mathbb{R}^{H_{\text{head}} \times H}$, $\mathbf{w}_2 \in \mathbb{R}^{H_{\text{head}}}$.

#### Variant B: LayerNorm Head (`classifier_norm="layernorm"`, `classifier_hidden=32/64`)

$$\tilde{\mathbf{h}} = \mathbf{W}_1 \mathbf{h}_{T_s} + \mathbf{b}_1 \in \mathbb{R}^{H_{\text{head}}}$$

$$\mathbf{h}_{\text{LN}} = \text{LayerNorm}(\tilde{\mathbf{h}}) = \frac{\tilde{\mathbf{h}} - \mu_{\tilde{h}}}{\sigma_{\tilde{h}} + \epsilon} \odot \boldsymbol{\gamma}_{\text{LN}} + \boldsymbol{\beta}_{\text{LN}}$$

$$z_{\text{logit}} = \mathbf{w}_2^\top \text{Dropout}\left(\text{ReLU}(\mathbf{h}_{\text{LN}})\right) + b_2$$

**Why LayerNorm (and not BatchNorm)?**
1. **Batch Size Invariance**: RNN batch sizes can be small ($B=16$), and subject trajectories have variable visit lengths. BatchNorm estimates batch statistics that degrade at small batch sizes and fail during single-subject inference ($B=1$).
2. **Logit Sharpening**: Standardizing activations across the hidden features prevents activation saturation, resulting in wider probability separation between converters and stable MCI.

#### Variant C: Direct Linear Head (`classifier_hidden=0`)

$$z_{\text{logit}} = \mathbf{w}^\top \mathbf{h}_{T_s} + b \quad (\text{where } \mathbf{w} \in \mathbb{R}^H)$$

Eliminates intermediate hidden activations and dropout, reducing head parameters to exactly $H + 1$ (e.g. 33 weights).

---

## 8. Ablation Axis 7: Capacity Scaling & Sequence Truncation

### 8.1 Capacity & Depth Grid

| Configuration | Recurrent Cell | Layers $L$ | Hidden $H$ | Head $H_{\text{head}}$ | Recurrent Params | Head Params | Active Trainable (Frozen Encoder) | Params / Subject ($n=133$) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Legacy Large** | LSTM | 2 | 128 | 64 | 231,936 | 8,321 | **240,257** | 1,806 |
| **Medium** | LSTM | 1 | 64 | 64 | 33,536 | 4,225 | **37,761** | 284 |
| **Standard (Headline)**| LSTM | 1 | 32 | 32 | 12,672 | 1,089 | **13,761** | 103 |
| **Minimal LSTM** | LSTM | 1 | 32 | 0 (Direct) | 12,672 | 33 | **12,705** | 96 |
| **Standard GRU** | GRU | 1 | 32 | 32 | 9,408 | 1,089 | **10,497** | 79 |
| **Minimal GRU** | GRU | 1 | 16 | 0 (Direct) | 3,984 | 17 | **4,001** | 30 |

---

### 8.2 Sequence Truncation: First-$N$ Visits & Window Controls
Early detection requires evaluating how early in disease progression a model can reliably predict conversion:
- `max_visits = 1`: Baseline scan only (static conversion risk).
- `max_visits = 2`: Baseline + Month 12 follow-up.
- `max_visits = 3`: Baseline + Month 12 + Month 24 follow-up.

#### Confounding Control: `require_full_window`
In raw datasets, converters often stay in clinical studies longer or undergo more follow-up visits, creating a **visit-count confound** (where sequence length $T$ alone correlates with $y$).
Setting `require_full_window=True` enforces that every retained subject has exactly $N$ scans, eliminating sequence-length leakage.

---

## 9. Cross-Paradigm Structural Comparison & Recurrent Architecture Map

### Diagram: Recurrent Core, Classifier Head, & Paradigm Benchmarks

![Figure 2: Recurrent Core, Head & Paradigm Ablations](fig2_gelstm_recurrent_head_and_paradigm_ablations.png)

---

### Detailed Paradigm Formulations:

The codebase implements three distinct longitudinal modeling paradigms over the GAAE graph embeddings:

| Paradigm | Architecture Type | Input Representation per Subject | Forward Operation | Target Inductive Bias |
| :--- | :--- | :--- | :--- | :--- |
| **geLSTM / geGRU** | Recurrent Sequence Model | Sequential list of pairs $(\hat{\mathbf{z}}_t, \Delta t_t)_{t=1}^{T_s}$ | Step-by-step non-linear gating via `PackedSequence` | Temporal order, causal progression, variable length |
| **GEC-MLP** | Flattened Fixed-Width MLP | Zero-padded flat vector $[\hat{\mathbf{z}}_1 \|\Delta t_1 \| \dots \| \hat{\mathbf{z}}_M \|\Delta t_M] \in \mathbb{R}^{390}$ | Multi-layer feedforward projection ($390 \to 256 \to 128 \to 64 \to 1$) | Joint cross-visit non-linear interactions |
| **GEP** | Static Mean-Pooled MLP | Single time-averaged vector $\bar{\mathbf{z}} = \frac{1}{T_s} \sum_{t=1}^{T_s} \hat{\mathbf{z}}_t \in \mathbb{R}^{64}$ | Compact MLP ($64 \to 32 \to 1$) | Temporal-invariance, subject-level mean connectome |

### Detailed Mathematical Formulations:

1. **geLSTM / geGRU**:
   Processes $(\mathbf{u}_1, \dots, \mathbf{u}_T)$ sequentially through non-linear recurrent gates:

   $$\mathbf{h}_t, \mathbf{c}_t = \text{LSTM}(\mathbf{u}_t, \mathbf{h}_{t-1}, \mathbf{c}_{t-1}), \quad \hat{y} = \sigma(\text{Head}(\mathbf{h}_{T_s}))$$

2. **GEC-MLP (Flattened Trajectory)**:
   Flattens the full visit history into a fixed-width vector padded with zeros up to $M=6$ visits:

   $$\mathbf{v}_{\text{flat}} = [\hat{\mathbf{z}}_1 \,\|\, \Delta t_1 \,\|\, \hat{\mathbf{z}}_2 \,\|\, \Delta t_2 \,\|\, \dots \,\|\, \hat{\mathbf{z}}_M \,\|\, \Delta t_M] \in \mathbb{R}^{M \times (d+1)} = \mathbb{R}^{6 \times 65} = \mathbb{R}^{390}$$

   $$\hat{y} = \text{MLP}_{390 \to 256 \to 128 \to 64 \to 1}(\mathbf{v}_{\text{flat}})$$

3. **GEP (Static Mean-Pooled MLP)**:
   Collapses the entire longitudinal trajectory into a single mean graph embedding:

   $$\bar{\mathbf{z}} = \frac{1}{T_s} \sum_{t=1}^{T_s} \hat{\mathbf{z}}_t \in \mathbb{R}^{64} \implies \hat{y} = \text{MLP}_{64 \to 32 \to 1}(\bar{\mathbf{z}})$$

---

## 10. Comprehensive Master Ablation Comparison Matrix

| Ablation Axis | Variant / Arm | Input Dimension | Recurrent Cell | Head Architecture | Active Trainable Params | Key Inductive Bias / Hypothesis |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Encoder Init** | `pretrained_frozen` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **13,761** | GAAE reconstruction features frozen; minimal downstream capacity |
| **Encoder Init** | `pretrained_finetuned`| $64 + 1 = 65$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **303,985** | Adapts reconstruction features to conversion classification |
| **Encoder Init** | `random` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **303,985** | Isolates GATv2 architecture inductive bias without pretraining |
| **Encoder Init** | `none` | $200 + 1 = 201$| LSTM ($H=32$) | Linear-ReLU-Dropout | **31,169** | Raw ROI pooling; tests if graph encoder earns its place |
| **Cell Type** | `gegru` | $64 + 1 = 65$ | GRU ($H=32$) | Linear-ReLU-Dropout | **10,497** | 3 gates, no cell state; optimal capacity for $T \le 6, N \approx 133$ |
| **Time Dynamics**| `with_time_delta` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **13,761** | Models non-uniform calendar intervals between visits |
| **Time Dynamics**| `no_time_delta` | $64$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **12,545** | Treats visit transitions as discrete uniform steps |
| **Time Dynamics**| `zero_time_delta` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **13,761** | Zeros temporal signal at test time without dimension shift |
| **Order Sanity** | `shuffle_order` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **13,761** | Random permutation of visits; tests directional dynamics |
| **Latent Filter**| `fdr_top_k` ($K=16$)| $16 + 1 = 17$ | LSTM ($H=32$) | Linear-ReLU-Dropout | **7,489** | Restricts recurrent input to top discriminative Fisher dimensions |
| **Head Norm** | `layernorm` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear-LN-ReLU-Dropout| **13,825** | Sharpens logits, widens output probability distribution |
| **Direct Head** | `linear_direct` | $64 + 1 = 65$ | LSTM ($H=32$) | Linear(32, 1) | **12,705** | Minimal classifier capacity without hidden layer |

---

## 11. Executable PyTorch Verification & Architecture Inspector

The executable cells below import the repository's native modules, programmatically instantiate every ablation model, print parameter breakdowns, and verify gradient flow and forward pass tensor shapes with synthetic data batches.

In [1]:
# Cell 1: Environment & Module Setup
import sys
from pathlib import Path

repo_root = Path('/mnt/e/fyassine/ad-early-detection')
classifier_root = repo_root / 'CLASSIFIER'

for p in (str(repo_root), str(classifier_root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_sequence

from CLASSIFIER.configs.encoder import encoder_arm, resolve_encoder_init, ENCODER_INIT_ARMS
from CLASSIFIER.configs.gelstm import GELSTMTrainConfig, EvalConfig
from CLASSIFIER.model.GELSTM.models import GELSTMClassifier, build_classifier_head

print("PyTorch Version:", torch.__version__)
print("Available Encoder Arms:", ENCODER_INIT_ARMS)

PyTorch Version: 2.10.0+cu128
Available Encoder Arms: ('pretrained_frozen', 'pretrained_finetuned', 'random', 'none')


In [2]:
# Cell 2: Automated Parameter Counting & Inspection Function
def inspect_model_architecture(model: GELSTMClassifier, name: str):
    total_params = sum(p.numel() for p in model.parameters())
    
    # Active encoder modules (GAT layers + FiLM) vs Decoder (unused in downstream forward pass)
    active_encoder_mods = model.encoder_modules()
    active_encoder_params = sum(sum(p.numel() for p in mod.parameters()) for mod in active_encoder_mods)
    active_encoder_trainable = sum(sum(p.numel() for p in mod.parameters() if p.requires_grad) for mod in active_encoder_mods)
    
    recurrent_params = sum(p.numel() for p in model.lstm.parameters())
    recurrent_trainable = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
    
    head_params = sum(p.numel() for p in model.classifier.parameters())
    head_trainable = sum(p.numel() for p in model.classifier.parameters() if p.requires_grad)
    
    # Active downstream trainable parameters
    active_downstream_trainable = active_encoder_trainable + recurrent_trainable + head_trainable
    
    print(f"============================================================")
    print(f"MODEL CONFIGURATION: {name}")
    print(f"============================================================")
    print(f" • Encoder Arm:            {model.encoder_init}")
    print(f" • Encoder Module Present: {model.encoder is not None}")
    print(f" • Embedding Dim:          {model.embed_dim}")
    print(f" • Recurrent Cell Type:    {model.rnn_type.upper()}")
    print(f" • Recurrent Input Dim:    {model.lstm_input_dim}")
    print(f" • Classifier Head Norm:   {model.classifier_norm}")
    print(f"------------------------------------------------------------")
    print(f" PARAMETER BREAKDOWN:")
    print(f"   - Active Encoder Params:{active_encoder_params:>8,d} (Trainable: {active_encoder_trainable:>8,d})")
    print(f"   - Recurrent Params:     {recurrent_params:>8,d} (Trainable: {recurrent_trainable:>8,d})")
    print(f"   - Classifier Head:      {head_params:>8,d} (Trainable: {head_trainable:>8,d})")
    print(f"   --------------------------------------------------------")
    print(f"   - ACTIVE TRAINABLE:     {active_downstream_trainable:>8,d}")
    print(f"   - TOTAL MODULE PARAMS:  {total_params:>8,d}")
    print(f"============================================================\n")

# Instantiate and inspect the 4 Encoder Arms
arms_to_test = [
    ("pretrained_frozen", "lstm", True, "none", True),
    ("pretrained_finetuned", "lstm", True, "none", False),
    ("random", "lstm", True, "none", False),
    ("none", "lstm", True, "none", False)
]

for arm, rnn, use_dt, head_norm, freeze in arms_to_test:
    m = GELSTMClassifier(
        in_features=200,
        gaae_hidden=200,
        gaae_latent=64,
        gaae_heads=1,
        gaae_cond_dim=2,
        gaae_dropout=0.0,
        lstm_hidden=32,
        lstm_layers=1,
        lstm_dropout=0.0,
        use_time_delta=use_dt,
        classifier_hidden=32,
        rnn_type=rnn,
        classifier_norm=head_norm,
        encoder_init=arm
    )
    if freeze:
        m.freeze_encoder()
    inspect_model_architecture(m, f"Arm: {arm} (Freeze={freeze})")

MODEL CONFIGURATION: Arm: pretrained_frozen (Freeze=True)
 • Encoder Arm:            pretrained_frozen
 • Encoder Module Present: True
 • Embedding Dim:          64
 • Recurrent Cell Type:    LSTM
 • Recurrent Input Dim:    65
 • Classifier Head Norm:   none
------------------------------------------------------------
 PARAMETER BREAKDOWN:
   - Active Encoder Params: 290,224 (Trainable:        0)
   - Recurrent Params:       12,672 (Trainable:   12,672)
   - Classifier Head:         1,089 (Trainable:    1,089)
   --------------------------------------------------------
   - ACTIVE TRAINABLE:       13,761
   - TOTAL MODULE PARAMS:   586,185

MODEL CONFIGURATION: Arm: pretrained_finetuned (Freeze=False)
 • Encoder Arm:            pretrained_finetuned
 • Encoder Module Present: True
 • Embedding Dim:          64
 • Recurrent Cell Type:    LSTM
 • Recurrent Input Dim:    65
 • Classifier Head Norm:   none
------------------------------------------------------------
 PARAMETER BREAKDOWN:
  

In [3]:
# Cell 3: Recurrent Cell & Head Normalization Comparison (LSTM vs GRU, Standard vs LayerNorm)
cell_configs = [
    ("GELSTM Standard", "lstm", 32, "none", "pretrained_frozen"),
    ("GEGRU Standard", "gru", 32, "none", "pretrained_frozen"),
    ("GELSTM LayerNorm Head", "lstm", 32, "layernorm", "pretrained_frozen"),
    ("GELSTM Direct Head (h=0)", "lstm", 0, "none", "pretrained_frozen"),
]

for name, rnn, head_hidden, norm, arm in cell_configs:
    m = GELSTMClassifier(
        in_features=200,
        gaae_hidden=200,
        gaae_latent=64,
        gaae_heads=1,
        gaae_cond_dim=2,
        gaae_dropout=0.0,
        lstm_hidden=32,
        lstm_layers=1,
        lstm_dropout=0.0,
        use_time_delta=True,
        classifier_hidden=head_hidden,
        rnn_type=rnn,
        classifier_norm=norm,
        encoder_init=arm
    )
    m.freeze_encoder()
    inspect_model_architecture(m, name)

MODEL CONFIGURATION: GELSTM Standard
 • Encoder Arm:            pretrained_frozen
 • Encoder Module Present: True
 • Embedding Dim:          64
 • Recurrent Cell Type:    LSTM
 • Recurrent Input Dim:    65
 • Classifier Head Norm:   none
------------------------------------------------------------
 PARAMETER BREAKDOWN:
   - Active Encoder Params: 290,224 (Trainable:        0)
   - Recurrent Params:       12,672 (Trainable:   12,672)
   - Classifier Head:         1,089 (Trainable:    1,089)
   --------------------------------------------------------
   - ACTIVE TRAINABLE:       13,761
   - TOTAL MODULE PARAMS:   586,185

MODEL CONFIGURATION: GEGRU Standard
 • Encoder Arm:            pretrained_frozen
 • Encoder Module Present: True
 • Embedding Dim:          64
 • Recurrent Cell Type:    GRU
 • Recurrent Input Dim:    65
 • Classifier Head Norm:   none
------------------------------------------------------------
 PARAMETER BREAKDOWN:
   - Active Encoder Params: 290,224 (Trainable:      

In [4]:
# Cell 4: Live Forward Pass & Tensor Dimensionality Verification
# Create synthetic batch of 3 subjects with variable visit counts (T1=4, T2=2, T3=1)
torch.manual_seed(42)
B = 3
seq_lens = [4, 2, 1]
embed_dim_standard = 64
use_dt = True
rnn_in_dim = embed_dim_standard + (1 if use_dt else 0)

# Simulate packed sequence embeddings [z_t || delta_t]
sequences = [torch.randn(length, rnn_in_dim) for length in seq_lens]
packed_batch = pack_sequence(sequences, enforce_sorted=True)

print(f"Created Synthetic PackedSequence for {B} subjects with lengths {seq_lens}")
print(f"Total time steps in batch: {packed_batch.data.shape[0]}, Feature width: {packed_batch.data.shape[1]}")

# Forward pass through GELSTM and GEGRU
models = {
    "GELSTM": GELSTMClassifier(200, 200, 64, 1, 2, 0.0, 32, 1, 0.0, use_time_delta=True, classifier_hidden=32, rnn_type="lstm"),
    "GEGRU": GELSTMClassifier(200, 200, 64, 1, 2, 0.0, 32, 1, 0.0, use_time_delta=True, classifier_hidden=32, rnn_type="gru")
}

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        logits = model(packed_batch)
        probs = torch.sigmoid(logits)
        print(f"[{name}] Forward Pass Output:")
        print(f"   Logits shape: {logits.shape} -> {logits.tolist()}")
        print(f"   Probabilities P(converter): {probs.tolist()}")

Created Synthetic PackedSequence for 3 subjects with lengths [4, 2, 1]
Total time steps in batch: 7, Feature width: 65


[GELSTM] Forward Pass Output:
   Logits shape: torch.Size([3]) -> [0.06260745227336884, 0.018213754519820213, 0.04956112429499626]
   Probabilities P(converter): [0.515646755695343, 0.5045533180236816, 0.512387752532959]
[GEGRU] Forward Pass Output:
   Logits shape: torch.Size([3]) -> [0.09239274263381958, 0.23306246101856232, 0.14025400578975677]
   Probabilities P(converter): [0.5230817794799805, 0.558003306388855, 0.5350061058998108]
